# Runbook 5.1.d: Data breaches and their impact  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Analyze Risk · **Maps to:** 5.1.C · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Build the severity scorer a security team's intake triage tool would run on each incoming breach
report. Given the categories of data exposed, the number of records affected, and how many days
it took to detect the breach, compute a total severity score and a label.

**Lesson objective.** Explain what a data breach is, identify the categories of data whose exposure causes the most harm, and estimate the severity of a breach from the type of data exposed, the number of records affected, and how quickly it was detected/contained.

### What comes in
`report`: `{"categories": list[str], "record_count": int, "days_to_detect": int}`

### What must come out
`{"score": int, "label": str}`

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
{"categories": ["public"], "record_count": 50, "days_to_detect": 1}
    -> {"score": 0, "label": "low"}

{"categories": ["pii", "auth_plaintext"], "record_count": 500000, "days_to_detect": 45}
    -> {"score": 40 + 40 + 20 + 20, "label": "critical"}   # = 120

{"categories": ["email"], "record_count": 200, "days_to_detect": 5}
    -> {"score": 10 + 10 + 10, "label": "medium"}          # = 30
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Multiple categories are additive; unknown category strings contribute 0 points.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(report: dict) -> dict
```

#### Input (`report`)
`{"categories": list[str], "record_count": int, "days_to_detect": int}`

- `categories`: zero or more of `{"pii", "financial", "health", "auth_plaintext", "auth_hashed",
  "email", "public"}`. Unknown strings are ignored (contribute 0 points).
- `record_count`: non-negative int.
- `days_to_detect`: non-negative int.

#### Output (`result`)
`{"score": int, "label": str}` where `label` is one of `"low"`, `"medium"`, `"high"`, `"critical"`.

#### Rubric
- Category points (sum, each category counted once): pii=40, financial=40, health=40,
  auth_plaintext=40, auth_hashed=15, email=10, public=0, unknown=0.
- Scale points: `<100`->0, `100-9999`->10, `10000-999999`->20, `>=1000000`->30.
- Detection points: `<=1`->0, `2-30`->10, `>30`->20.
- Label thresholds on total score: `>=80`->critical, `>=50`->high, `>=20`->medium, else low.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Categories with duplicates in the input list are only counted once.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import Dict

CATEGORY_POINTS = {
    "pii": 40,
    "financial": 40,
    "health": 40,
    "auth_plaintext": 40,
    "auth_hashed": 15,
    "email": 10,
    "public": 0,
}


def _scale_points(record_count: int) -> int:
    if record_count >= 1_000_000:
        return 30
    if record_count >= 10_000:
        return 20
    if record_count >= 100:
        return 10
    return 0


def _detect_points(days_to_detect: int) -> int:
    if days_to_detect > 30:
        return 20
    if days_to_detect >= 2:
        return 10
    return 0


def _label(score: int) -> str:
    if score >= 80:
        return "critical"
    if score >= 50:
        return "high"
    if score >= 20:
        return "medium"
    return "low"


def solve(report: Dict[str, object]) -> Dict[str, object]:
    categories = report.get("categories", [])
    record_count = report.get("record_count", 0)
    days_to_detect = report.get("days_to_detect", 0)

    score = sum(CATEGORY_POINTS.get(cat, 0) for cat in categories)
    score += _scale_points(record_count)
    score += _detect_points(days_to_detect)

    return {"score": score, "label": _label(score)}


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_low_public_small():
    report = {"categories": ["public"], "record_count": 50, "days_to_detect": 1}
    assert impl.solve(report) == {"score": 0, "label": "low"}


def test_critical_pii_plaintext_large_slow():
    report = {"categories": ["pii", "auth_plaintext"], "record_count": 500000, "days_to_detect": 45}
    assert impl.solve(report) == {"score": 120, "label": "critical"}


def test_medium_email_small_moderate():
    report = {"categories": ["email"], "record_count": 200, "days_to_detect": 5}
    assert impl.solve(report) == {"score": 30, "label": "medium"}


def test_hashed_auth_only():
    report = {"categories": ["auth_hashed"], "record_count": 50, "days_to_detect": 1}
    assert impl.solve(report) == {"score": 15, "label": "low"}


def test_high_financial_million_records_fast_detect():
    report = {"categories": ["financial"], "record_count": 2_000_000, "days_to_detect": 1}
    assert impl.solve(report) == {"score": 70, "label": "high"}


def test_multiple_categories_additive():
    report = {"categories": ["pii", "financial", "health"], "record_count": 50, "days_to_detect": 1}
    assert impl.solve(report) == {"score": 120, "label": "critical"}


def test_unknown_category_contributes_zero():
    report = {"categories": ["mystery_category"], "record_count": 50, "days_to_detect": 1}
    assert impl.solve(report) == {"score": 0, "label": "low"}


def test_empty_report():
    report = {"categories": [], "record_count": 0, "days_to_detect": 0}
    assert impl.solve(report) == {"score": 0, "label": "low"}


def test_boundary_record_count_10000():
    report = {"categories": [], "record_count": 10000, "days_to_detect": 0}
    assert impl.solve(report) == {"score": 20, "label": "medium"}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve({"categories": ["public"], "record_count": 50, "days_to_detect": 1}),
     {"score": 0, "label": "low"}),
    (lambda: solve({"categories": ["pii", "auth_plaintext"], "record_count": 500000, "days_to_detect": 45}),
     {"score": 40 + 40 + 20 + 20, "label": "critical"}),
    (lambda: solve({"categories": ["email"], "record_count": 200, "days_to_detect": 5}),
     {"score": 10 + 10 + 10, "label": "medium"}),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it